# Módszertani kódkivonat – parlagfű detektálása Sentinel–2 idősorok alapján

Ez a notebook a diplomamunkában bemutatott feldolgozási lánc főbb Python-alapú lépéseit dokumentálja. 
A kód lokális munkakörnyezetben készült, ezért az útvonalak és bemeneti állományok a reprodukcióhoz módosítandók.

A notebook fő lépései:
1. Sentinel–2 L2A képek lekérdezése és letöltése a Copernicus Data Space Ecosystem API-n keresztül.
2. T33TYM és T33TYN MGRS csempék mozaikolása.
3. Fejér vármegyei mintaterület kivágása.
4. A sávok 10 méteres rácsra illesztése.
5. SCL-alapú felhőmaszkolás és 10 napos medián kompozitok készítése.
6. Jellemzők előállítása a tanítóadatokhoz
7. Random Forest modell tanítása és kiértékelése.
8. Klasszifikált parlagfű-raszter előállítása.

Megjegyzés: a notebook nem önállóan reprodukálható csomag, hanem a dolgozatban ismertetett módszertan programkód-szintű dokumentációja.

1. Sentinel-2 L2A képek letöltése

In [ ]:
from datetime import date, timedelta
import os
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

#Beállítások 
copernicus_user = "sajat_e-mail"
copernicus_password = "sajat_jelszo"
ft = "POLYGON ((18.973846457524814 47.612750363213905, 18.02257850018782 47.612750363213905, 18.02257850018782 46.67139790548728, 18.973846457524814 46.67139790548728, 18.973846457524814 47.612750363213905))" #Letölteni kívánt terület BBox WKT
data_collection = "SENTINEL-2"
s_d = "2025-01-30"  #első nap
e_d = "2025-10-01"  #utolsó nap
wanted_tiles = ["T33TYN", "T33TYM"] #letöltendő MGRS csempék
output_folder = "/Volumes/T7 Touch/06_szakdoga_sentinel/2025_raw" #célmappa a letöltéshez

#token lekérése
def get_keycloak(username: str, password: str) -> str:
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data=data,
    )
    r.raise_for_status()
    return r.json()["access_token"]
    
# Adatok lekérdezése:
response = requests.get(
    f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?"
    f"$filter=Collection/Name eq '{data_collection}' "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{ft}') "
    f"and ContentDate/Start gt {s_d}T00:00:00.000Z "
    f"and ContentDate/Start lt {e_d}T00:00:00.000Z"
    f"&$count=True&$top=1000"
)

if response.status_code != 200:
    print(f"Hiba az API lekérdezés során: {response.status_code} - {response.text}")
else:
    json_ = response.json()
    products = json_.get("value", [])
    
    p = pd.DataFrame.from_dict(products)
    if p.shape[0] == 0:
        print("Nincs találat a megadott időszakra és területre.")
    else:
        p["geometry"] = p["GeoFootprint"].apply(shape)
        productDF = gpd.GeoDataFrame(p).set_geometry("geometry")
        productDF = productDF[~productDF["Name"].str.contains("L1C")]
        productDF = productDF[productDF["Name"].str.contains("|".join(wanted_tiles))]
        productDF["identifier"] = productDF["Name"].str.split(".").str[0]
        print(f"Összesen {len(productDF)} L2A csempe található a szűrés után.")

        for index, feat in enumerate(productDF.iterfeatures()):
            props = feat["properties"]
            name = props["Name"]
            identifier = props["identifier"]
            save_path = os.path.join(output_folder, f"{identifier}.zip")
            
            if os.path.exists(save_path):
                print(f"A(z) {identifier} fájl már létezik, kihagyva.")
                continue

            print(f"Letöltés: {name}")
            try:
                session = requests.Session()
                keycloak_token = get_keycloak(copernicus_user, copernicus_password)
                session.headers.update({"Authorization": f"Bearer {keycloak_token}"})

                url = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products({props['Id']})/$value"
                response = session.get(url, allow_redirects=False)
                while response.status_code in (301, 302, 303, 307):
                    url = response.headers["Location"]
                    response = session.get(url, allow_redirects=False)

                file = session.get(url, verify=False, allow_redirects=True)
                save_path = os.path.join(output_folder, f"{props['identifier']}.zip")

                with open(save_path, "wb") as f:
                    f.write(file.content)
                print(f"Mentve: {save_path}")

            except Exception as e:
                print(f"Hiba a letöltésnél: {e}")


2. MGRS csempék mozaikolása

In [ ]:
import os
import glob
import zipfile
import shutil
import time
import rasterio
from rasterio.merge import merge
from collections import defaultdict

#Beállítások
input_folder = "/Volumes/T7 Touch/06_szakdoga_sentinel/2025_raw"
output_folder = "/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged"
os.makedirs(output_folder, exist_ok=True)

tiles = ["T33TYN", "T33TYM"]

bands_10m = ["B02", "B03", "B04", "B08"]
bands_20m = ["B05", "B06", "B07", "B8A", "B11", "B12"]
bands_60m = ["B01", "B09", "B10"]

# Segédfüggvény a sávok keresésére
def find_band_path(safe_dir, tile, date, band, res):
    pattern = os.path.join(
        safe_dir, "GRANULE", "*", "IMG_DATA", "**",
        f"{tile}_{date}T*_{band}_{res}m.jp2"
    )
    matches = glob.glob(pattern, recursive=True)
    if not matches:
        pattern_alt = os.path.join(safe_dir, "**", f"{band}_{res}m.jp2")
        matches = glob.glob(pattern_alt, recursive=True)
    return matches[0] if matches else None

# Végtelen ciklus a folyamatos feldolgozásra
while True:
    all_zips = glob.glob(os.path.join(input_folder, "*.zip"))
    
    for z in all_zips:
        base_name = os.path.splitext(os.path.basename(z))[0]
        safe_dir = os.path.join(input_folder, base_name + ".SAFE")
        if not os.path.exists(safe_dir):
            with zipfile.ZipFile(z, 'r') as zip_ref:
                zip_ref.extractall(input_folder)
                print(f"Kicsomagolva: {safe_dir}")
            os.remove(z)
            print(f"Törölve ZIP: {z}")

        safe_dict = defaultdict(dict)
        all_safe = glob.glob(os.path.join(input_folder, "*.SAFE"))
        for safe_path in all_safe:
            fname = os.path.basename(safe_path)
            parts = fname.split("_")
            date_part = parts[2][:8]  # YYYYMMDD
            for t in tiles:
                if t in fname:
                    safe_dict[date_part][t] = safe_path

        for date_key in sorted(safe_dict.keys()):
            tile_pair = safe_dict[date_key]
            if not all(t in tile_pair for t in tiles):
                continue  # mindkét tile kell a merge-hez

            out_dir = os.path.join(output_folder, f"{date_key}_T33TYM-TYN")
            os.makedirs(out_dir, exist_ok=True)

            # Merge minden sávra
            for band_group, res in zip([bands_10m, bands_20m, bands_60m], ["10", "20", "60"]):
                for band in band_group:
                    band_files = []
                    for tile in tiles:
                        band_path = find_band_path(tile_pair[tile], tile, date_key, band, res)
                        if band_path:
                            band_files.append(band_path)
                        else:
                            print(f"Band {band} not found in {tile} at R{res}m")
                    if len(band_files) == 2:
                        srcs = [rasterio.open(f) for f in band_files]
                        merged, out_trans = merge(srcs)
                        out_meta = srcs[0].meta.copy()
                        out_meta.update({
                            "height": merged.shape[1],
                            "width": merged.shape[2],
                            "transform": out_trans,
                            "count": 1
                        })
                        out_path = os.path.join(out_dir, f"{band}.tif")
                        with rasterio.open(out_path, "w", **out_meta) as dest:
                            dest.write(merged[0], 1)
                        for s in srcs:
                            s.close()
                        print(f"Saved merged band {band} ({res}m) -> {out_path}")

            # TCI 60m merge
            tci_files = []
            for tile in tiles:
                tci_path = find_band_path(tile_pair[tile], tile, date_key, "TCI", "60")
                if tci_path:
                    tci_files.append(tci_path)
            if len(tci_files) == 2:
                srcs = [rasterio.open(f) for f in tci_files]
                merged, out_trans = merge(srcs)
                out_meta = srcs[0].meta.copy()
                out_meta.update({
                    "height": merged.shape[1],
                    "width": merged.shape[2],
                    "transform": out_trans,
                    "count": 3  # TCI RGB
                })
                tci_out_path = os.path.join(out_dir, "TCI.tif")
                with rasterio.open(tci_out_path, "w", **out_meta) as dest:
                    for i in range(3):
                        dest.write(merged[i], i+1)
                for s in srcs:
                    s.close()
                print(f"Saved merged TCI -> {tci_out_path}")

            # SCL 20m merge
            scl_files = []
            for tile in tiles:
                scl_path = find_band_path(tile_pair[tile], tile, date_key, "SCL", "20")
                if scl_path:
                    scl_files.append(scl_path)
            if len(scl_files) == 2:
                srcs = [rasterio.open(f) for f in scl_files]
                merged, out_trans = merge(srcs)
                out_meta = srcs[0].meta.copy()
                out_meta.update({
                    "height": merged.shape[1],
                    "width": merged.shape[2],
                    "transform": out_trans,
                    "count": 1
                })
                scl_out_path = os.path.join(out_dir, "SCL.tif")
                with rasterio.open(scl_out_path, "w", **out_meta) as dest:
                    dest.write(merged[0], 1)
                for s in srcs:
                    s.close()
                print(f"Saved merged SCL -> {scl_out_path}")

            for t in tiles:
                if os.path.exists(tile_pair[t]):
                    shutil.rmtree(tile_pair[t])
                    print(f"Törölve SAFE: {tile_pair[t]}")

    print("Várakozás 5 percig a következő ellenőrzésig...")
    time.sleep(300)


3. Mintaterület kivágása

In [ ]:
import os
import glob
import rasterio
from rasterio.mask import mask
import geopandas as gpd

# Beállítások
input_folder = "/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged"
output_folder = "/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb"
shapefile_path = "/Users/petrascsaba/Library/CloudStorage/OneDrive-EotvosLorandTudomanyegyetemInformatikaiKar/Pál Márton fájljai - petras_csaba/02_input_data/fejer_bb.shp"
megye_name = "fejer"
tci = ["TCI"]

shape = gpd.read_file(shapefile_path)
tci_folder = os.path.join(output_folder, "TCI_clipped")
os.makedirs(tci_folder, exist_ok=True)

# Almappák kezelése
for folder in sorted(os.listdir(input_folder)):
    folder_path = os.path.join(input_folder, folder)
    if not os.path.isdir(folder_path) or "_" not in folder:
        continue
    date_part = folder.split("_")[0]
    print(f"\n Feldolgozás: {folder} ({date_part})")

    out_subfolder = os.path.join(output_folder, f"{date_part}_{megye_name}")
    os.makedirs(out_subfolder, exist_ok=True)

    tif_files = sorted(glob.glob(os.path.join(folder_path, "*.tif")))
    if not tif_files:
        print("Nincsenek TIFF fájlok, kihagyva.")
        continue

    for tif in tif_files:
        fname = os.path.basename(tif)
        try:
            with rasterio.open(tif) as src:
                shape_proj = shape.to_crs(src.crs) if shape.crs != src.crs else shape
                clipped, out_transform = mask(src, shapes=shape_proj.geometry, crop=True)
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": clipped.shape[1],
                    "width": clipped.shape[2],
                    "transform": out_transform,
                    "dtype": src.dtypes[0]
                })

                # TCI fájlok kezelése
                if any(k.lower() in fname.lower() for k in tci):
                    out_meta.update({
                        "compress": "ZSTD",
                        "predictor": 2,
                        
                    })
                    out_dir = tci_folder
                    clip_type = "tömörített TCI"
                else:
                    out_dir = out_subfolder
                    clip_type = "normál clip"

                os.makedirs(out_dir, exist_ok=True)

                band_name = fname.split("_")[-1].replace(".tif", "")
                out_name = f"{date_part}_{band_name}.tif"
                out_path = os.path.join(out_dir, out_name)

                # Fájl mentése
                with rasterio.open(out_path, "w", **out_meta) as dest:
                    dest.write(clipped)

                print(f"{out_name} mentve ({clip_type}).")

        except Exception as e:
            print(f"Hiba a fájl feldolgozásánál: {fname}\n     {e}")

print("\nFeldolgozás befejezve!")


4. 10 m-es újramintavételezés

In [ ]:
import os
import glob
import shutil
import rasterio
from rasterio.enums import Resampling

base_dir = '/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb'
out_base_dir = '/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb_10m'

os.makedirs(out_base_dir, exist_ok=True)

date_folders = glob.glob(os.path.join(base_dir, '*_fejer'))

for folder in date_folders:
    print(f"\nFeldolgozás alatt: {folder}")

    folder_name = os.path.basename(os.path.normpath(folder))  # pl. 20250730_fejer

    ref_files = glob.glob(os.path.join(folder, "*_B02.tif")) + glob.glob(os.path.join(folder, "*_B2.tif"))
    if not ref_files:
        print("Hiba: Nem található 10 m-es referencia (B02) a mappában.")
        continue

    ref_path = ref_files[0]

    with rasterio.open(ref_path) as ref_src:
        ref_meta = ref_src.meta.copy()
        ref_transform = ref_src.transform
        ref_width = ref_src.width
        ref_height = ref_src.height
        ref_crs = ref_src.crs

    all_tifs = glob.glob(os.path.join(folder, "*.tif"))

    for tif in all_tifs:
        filename = os.path.basename(tif)
        out_filename = f"{filename}"
        out_path = os.path.join(out_base_dir, out_filename)

        if os.path.exists(out_path):
            print(f"  Ugrás: {out_filename} már létezik.")
            continue

        with rasterio.open(tif) as src:
            same_grid = (
                src.width == ref_width and
                src.height == ref_height and
                src.transform == ref_transform and
                src.crs == ref_crs
            )
            if same_grid:
                shutil.copy2(tif, out_path)
                print(f"  Másolva: {out_filename}")
                continue

            # SCL: nearest, minden más: bilinear
            if 'SCL' in filename.upper():
                resampling_method = Resampling.nearest
            else:
                resampling_method = Resampling.bilinear

            data = src.read(
                1,
                out_shape=(ref_height, ref_width),
                resampling=resampling_method
            )

            out_meta = src.meta.copy()
            out_meta.update({
                "crs": ref_crs,
                "transform": ref_transform,
                "width": ref_width,
                "height": ref_height,
            })

            with rasterio.open(out_path, 'w', **out_meta) as dst:
                dst.write(data, 1)

            print(f"  Újramintavételezve: {out_filename}")

print("\nKész!")

5. 10 napos Sentinel-2 kompozitok készítése

In [ ]:
import os
import glob
import gc
from collections import defaultdict
from datetime import datetime, timedelta

import numpy as np
import rasterio

# Beállítások


INPUT_DIR = '/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb_10m'
OUTPUT_DIR = '/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb_10m_composites'

VALID_SCL_CLASSES = [4, 5, 6, 7]
BANDS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']

MIN_VALID_RATIO = 0.60  # minimum használható pixelarány (felhősség)

os.makedirs(OUTPUT_DIR, exist_ok=True)

ALL_TOKENS = ['SCL', 'B8A', 'B12', 'B11', 'B08', 'B07', 'B06', 'B05', 'B04', 'B03', 'B02']

#1. Segédfüggvények


def get_10_day_period(date_str: str) -> str:
    dt = datetime.strptime(date_str, '%Y%m%d')
    doy_0 = dt.timetuple().tm_yday - 1
    period_idx = doy_0 // 10
    start_date = datetime(dt.year, 1, 1) + timedelta(days=period_idx * 10)
    end_date = start_date + timedelta(days=9)

    if start_date.year != end_date.year:
        end_date = datetime(dt.year, 12, 31)

    return f"{start_date.strftime('%Y%m%d')}_{end_date.strftime('%Y%m%d')}"


def extract_date_and_token(filename: str):
    name = os.path.splitext(os.path.basename(filename))[0]
    parts = name.split('_')

    if len(parts) < 2:
        return None, None

    date_str = parts[0]
    upper_name = name.upper()

    try:
        datetime.strptime(date_str, '%Y%m%d')
    except ValueError:
        return None, None

    token_found = None
    for token in ALL_TOKENS:
        if f"_{token}" in upper_name or upper_name.endswith(token):
            token_found = token
            break

    return date_str, token_found

#2. Input fájlok indexelése


files_by_date = defaultdict(dict)

all_tifs = glob.glob(os.path.join(INPUT_DIR, '*.tif'))

for filepath in all_tifs:
    filename = os.path.basename(filepath)

    if '_composite' in filename.lower():
        continue

    date_str, token = extract_date_and_token(filename)

    if date_str is None or token is None:
        print(f"Kihagyva (nem megfelelő fájlnév): {filename}")
        continue

    if token in files_by_date[date_str]:
        print(f"Figyelem: duplikált fájl ugyanarra a dátumra és tokenre: {filename}")
        print(f"  Meglévő: {files_by_date[date_str][token]}")
        print(f"  Új:      {filepath}")
        print("  Az új fájl felülírja a régit az indexben.\n")

    files_by_date[date_str][token] = filepath

print(f"\nÖsszes feldolgozható dátum: {len(files_by_date)}")


# 3. Dátumok csoportosítása 10 napos periódusokba
dates_by_period = defaultdict(list)

for date_str in sorted(files_by_date.keys()):
    try:
        period_key = get_10_day_period(date_str)
        dates_by_period[period_key].append(date_str)
    except Exception as e:
        print(f"Hiba a dátum periódusba sorolásakor: {date_str} -> {e}")


# 4. Kompozitképzés periódusonként
for period, date_list in sorted(dates_by_period.items()):
    print(f"\nFeldolgozás: {period} ({len(date_list)} dátum)")

    masks_by_date = {}
    meta = None
    accepted_dates = []

    # --- SCL maszkok előkészítése és felhős jelenetek szűrése ---
    for date_str in date_list:
        scl_path = files_by_date[date_str].get('SCL')

        if scl_path is None:
            print(f"  Nincs SCL ehhez a dátumhoz, kihagyva: {date_str}")
            continue

        try:
            with rasterio.open(scl_path) as src:
                scl = src.read(1)
                mask = np.isin(scl, VALID_SCL_CLASSES)

                valid_ratio = np.sum(mask) / mask.size
                print(f"  {date_str} - érvényes pixelarány: {valid_ratio:.2%}")

                if valid_ratio < MIN_VALID_RATIO:
                    print(f"    -> Túl felhős jelenet, kihagyva")
                    continue

                masks_by_date[date_str] = mask
                accepted_dates.append(date_str)

                if meta is None:
                    meta = src.meta.copy()

        except Exception as e:
            print(f"  Hiba az SCL beolvasásakor ({date_str}): {e}")

    if meta is None or not accepted_dates:
        print("  Nem maradt használható dátum ebben a periódusban, periódus kihagyva.")
        continue

    # --- Kompozit készítése sávonként ---
    for band in BANDS:
        band_data_stack = []

        for date_str in accepted_dates:
            band_path = files_by_date[date_str].get(band)
            mask = masks_by_date.get(date_str)

            if band_path is None or mask is None:
                continue

            try:
                with rasterio.open(band_path) as src:
                    arr = src.read(1).astype(np.float32)

                    if src.nodata is not None:
                        arr[arr == src.nodata] = np.nan

                    if arr.shape != mask.shape:
                        print(f"  Mérethiba: {date_str} - {band}, kihagyva.")
                        continue

                    arr[~mask] = np.nan
                    band_data_stack.append(arr)

            except Exception as e:
                print(f"  Hiba a sáv beolvasásakor ({date_str} - {band}): {e}")

        if not band_data_stack:
            print(f"  Nincs felhasználható adat ehhez a sávhoz: {band}")
            continue

        print(f"  Számítás és mentés: {band}...")

        try:
            with np.errstate(all='ignore'):
                result = np.nanmedian(np.stack(band_data_stack), axis=0)

            out_path = os.path.join(OUTPUT_DIR, f"{period}_{band}_composite.tif")

            out_meta = meta.copy()
            out_meta.update(
                dtype=rasterio.float32,
                count=1,
                nodata=np.nan
            )

            with rasterio.open(out_path, 'w', **out_meta) as dst:
                dst.write(result.astype(np.float32), 1)

        except Exception as e:
            print(f"  Hiba a kompozit mentésekor ({period} - {band}): {e}")

        del band_data_stack
        del result
        gc.collect()

    del masks_by_date
    gc.collect()

print("\nKész!")

6. Jellemzők előállítása

In [ ]:
import glob
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

# 1. Beállítások

COMPOSITE_DIR = Path("/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb_10m_composites")
SHAPEFILE = Path("/Users/petrascsaba/Library/CloudStorage/OneDrive-EotvosLorandTudomanyegyetemInformatikaiKar/Pál Márton fájljai - petras_csaba/02_input_data/parlagfu_parcellak.shp")
OUT_PARQUET = Path("/Users/petrascsaba/Library/CloudStorage/OneDrive-EotvosLorandTudomanyegyetemInformatikaiKar/Pál Márton fájljai - petras_csaba/03_output_data/01_RF_model/00_train/v1_2025_parlagfu.parquet")

ID_COL = "id"
LABEL_COL = "label"
TARGET_MAP = {0: 0, 1: 1, 2: 1}  # label 1 és 2 pozitív parlagfűként kezelve

BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
MONTH_MIN = 3
MONTH_MAX = 10
MODEL_MONTHS = ["05", "06", "07", "08", "09", "10"]
INDICES = ["NDVI", "GNDVI", "NDRE", "PSRI", "NDWI"]
STATS = ["min", "mean", "max", "std", "amplitude"]

DIFF_FEATURES = [
    "DIFF_NDVI_JUL_JUN", "DIFF_GNDVI_JUL_JUN", "DIFF_NDRE_JUL_JUN",
    "DIFF_NDVI_AUG_JUL", "DIFF_NDRE_AUG_JUL",
    "DIFF_NDVI_SEP_AUG", "DIFF_NDRE_SEP_AUG", "DIFF_PSRI_SEP_AUG",
    "RATIO_G_N_AUG", "RATIO_NDRE_NDVI_AUG",
]

PHENO_FEATURES = [
    "NDVI_DROP_SUMMER", "NDVI_RECOVERY_AUTUMN", "GNDVI_RECOVERY_AUTUMN",
    "NDRE_RECOVERY_AUTUMN", "SUMMER_TO_AUTUMN_SLOPE_NDVI",
    "SUMMER_TO_AUTUMN_SLOPE_GNDVI", "SUMMER_TO_AUTUMN_SLOPE_PSRI",
    "ARGMIN_MONTH_NDVI", "ARGMAX_MONTH_GNDVI",
]

META_COLS = ["id", "pixel_id", "target", "target_binary"]

# 2. Segédfüggvények

def find_composite_rasters():
    """A kompozit mappából összegyűjti a dátum-sáv párokat."""
    rasters = {}
    files = glob.glob(str(COMPOSITE_DIR / "*_composite.tif"))

    for file in files:
        stem = Path(file).stem
        parts = stem.split("_")
        period = parts[0]  # pl. 20250829
        band = None

        for p in parts:
            if p in BANDS:
                band = p
                break

        if band is None or len(period) != 8 or not period.isdigit():
            continue

        month = int(period[4:6])
        if MONTH_MIN <= month <= MONTH_MAX:
            if period not in rasters:
                rasters[period] = {}
            rasters[period][band] = file

    periods = sorted(rasters.keys())
    if not periods:
        raise RuntimeError("Nem találtam használható Sentinel-2 kompozitokat.")

    return rasters, periods


def get_reference_raster(rasters, periods):
    """Referencia raszter: lehetőleg B04, ha nincs, akkor az első elérhető sáv."""
    first_period = periods[0]
    if "B04" in rasters[first_period]:
        return rasters[first_period]["B04"]
    return list(rasters[first_period].values())[0]



def extract_raw_pixels(rasters, periods, ref_path):
    print("Tanítópoligonok beolvasása...")
    gdf = gpd.read_file(SHAPEFILE)

    with rasterio.open(ref_path) as ref:
        raster_crs = ref.crs

    gdf = gdf.to_crs(raster_crs).explode(index_parts=False).reset_index(drop=True)
    print(f"Poligonok száma: {len(gdf)}")

    all_records = []

    for i, row in gdf.iterrows():
        geom = [row.geometry]
        poly_id = row[ID_COL]
        label = row[LABEL_COL]

        # referencia alapján valid pixelmaszk
        with rasterio.open(ref_path) as src:
            try:
                ref_img, _ = mask(src, geom, crop=True, all_touched=True)
            except ValueError:
                continue

            ref_arr = ref_img[0]
            if src.nodata is not None:
                valid_mask = ref_arr != src.nodata
            else:
                valid_mask = np.isfinite(ref_arr)

        n_pixels = int(valid_mask.sum())
        if n_pixels == 0:
            continue

        data = {
            "id": [poly_id] * n_pixels,
            "pixel_id": np.arange(n_pixels),
            "target": [label] * n_pixels,
        }

        # minden dátum és sáv kinyerése
        for period in periods:
            for band in BANDS:
                col = f"{period}_{band}"
                path = rasters.get(period, {}).get(band)

                if path is None:
                    data[col] = np.full(n_pixels, np.nan)
                    continue

                with rasterio.open(path) as src:
                    try:
                        img, _ = mask(src, geom, crop=True, all_touched=True)
                        vals = img[0][valid_mask].astype("float32")
                        if src.nodata is not None:
                            vals[vals == src.nodata] = np.nan
                        data[col] = vals / 10000.0
                    except Exception:
                        data[col] = np.full(n_pixels, np.nan)

        all_records.append(pd.DataFrame(data))

        if (i + 1) % 25 == 0:
            print(f"Feldolgozott poligon: {i + 1}/{len(gdf)}")

    if not all_records:
        raise RuntimeError("Nem jött létre pixelrekord.")

    raw = pd.concat(all_records, ignore_index=True)
    raw["target_binary"] = raw["target"].map(TARGET_MAP)
    print(f"Nyers pixelmátrix mérete: {raw.shape}")
    return raw


def interpolate_bands(raw, periods):
    """Hiányzó sávértékek lineáris interpolációja időben, sávonként."""
    out = raw.copy()
    for band in BANDS:
        cols = [f"{p}_{band}" for p in periods if f"{p}_{band}" in out.columns]
        if cols:
            out[cols] = out[cols].interpolate(axis=1, method="linear", limit_direction="both")
    return out


def valid_rows(raw, periods):
    """Csak azok a pixelek maradnak, ahol minden sávhoz van legalább egy időpont."""
    ok = raw["target_binary"].notna()
    for band in BANDS:
        cols = [f"{p}_{band}" for p in periods]
        cols = [c for c in cols if c in raw.columns]
        ok = ok & raw[cols].notna().any(axis=1)
    return ok


def calculate_indices(raw, periods):
    """NDVI, GNDVI, NDRE, PSRI és NDWI számítása."""
    feat = pd.DataFrame(index=raw.index)

    with np.errstate(divide="ignore", invalid="ignore"):
        for p in periods:
            b2 = raw[f"{p}_B02"]
            b3 = raw[f"{p}_B03"]
            b4 = raw[f"{p}_B04"]
            b5 = raw[f"{p}_B05"]
            b6 = raw[f"{p}_B06"]
            b8 = raw[f"{p}_B08"]
            b11 = raw[f"{p}_B11"]

            feat[f"{p}_NDVI"] = (b8 - b4) / (b8 + b4)
            feat[f"{p}_GNDVI"] = (b8 - b3) / (b8 + b3)
            feat[f"{p}_NDRE"] = (b8 - b5) / (b8 + b5)
            feat[f"{p}_PSRI"] = (b4 - b2) / b6
            feat[f"{p}_NDWI"] = (b8 - b11) / (b8 + b11)

    return feat.replace([np.inf, -np.inf], np.nan)


def add_index_statistics(feat):
    """Indexenként min, max, átlag, szórás, amplitúdó."""
    out = feat.copy()

    for index_name in INDICES:
        cols = [c for c in out.columns if c.endswith(f"_{index_name}")]
        if not cols:
            continue

        values = out[cols]
        out[f"STAT_{index_name}_min"] = values.min(axis=1)
        out[f"STAT_{index_name}_mean"] = values.mean(axis=1)
        out[f"STAT_{index_name}_max"] = values.max(axis=1)
        out[f"STAT_{index_name}_std"] = values.std(axis=1)
        out[f"STAT_{index_name}_amplitude"] = values.max(axis=1) - values.min(axis=1)

    return out


def first_month_column(df, month, index_name):
    """Egy hónap első elérhető kompozitját adja vissza adott indexből."""
    cols = [c for c in df.columns if len(c) >= 8 and c[:8].isdigit() and c[4:6] == month and c.endswith(f"_{index_name}")]
    if not cols:
        return None
    return df[sorted(cols)[0]]


def add_differences(feat):
    """Fenológiai különbségek és arányok előállítása."""
    out = feat.copy()

    pairs = {
        "DIFF_NDVI_JUL_JUN": ("07", "06", "NDVI"),
        "DIFF_GNDVI_JUL_JUN": ("07", "06", "GNDVI"),
        "DIFF_NDRE_JUL_JUN": ("07", "06", "NDRE"),
        "DIFF_NDVI_AUG_JUL": ("08", "07", "NDVI"),
        "DIFF_NDRE_AUG_JUL": ("08", "07", "NDRE"),
        "DIFF_NDVI_SEP_AUG": ("09", "08", "NDVI"),
        "DIFF_NDRE_SEP_AUG": ("09", "08", "NDRE"),
        "DIFF_PSRI_SEP_AUG": ("09", "08", "PSRI"),
    }

    for name, (late_m, early_m, index_name) in pairs.items():
        late = first_month_column(out, late_m, index_name)
        early = first_month_column(out, early_m, index_name)
        if late is not None and early is not None:
            out[name] = late - early

    gndvi_aug = first_month_column(out, "08", "GNDVI")
    ndvi_aug = first_month_column(out, "08", "NDVI")
    ndre_aug = first_month_column(out, "08", "NDRE")

    if gndvi_aug is not None and ndvi_aug is not None:
        out["RATIO_G_N_AUG"] = gndvi_aug / (ndvi_aug + 1e-6)
    if ndre_aug is not None and ndvi_aug is not None:
        out["RATIO_NDRE_NDVI_AUG"] = ndre_aug / (ndvi_aug + 1e-6)

    return out


def month_columns(df, months, index_name):
    return [
        c for c in df.columns
        if len(c) >= 8 and c[:8].isdigit() and c[4:6] in months and c.endswith(f"_{index_name}")
    ]


def mean_by_months(df, months, index_name):
    cols = month_columns(df, months, index_name)
    if not cols:
        return None
    return df[cols].mean(axis=1)


def min_by_months(df, months, index_name):
    cols = month_columns(df, months, index_name)
    if not cols:
        return None
    return df[cols].min(axis=1)


def add_phenology(feat):
    """Néhány egyszerű, fenológiát leíró változó."""
    out = feat.copy()

    may_jun_ndvi = mean_by_months(out, ["05", "06"], "NDVI")
    jul_aug_ndvi_min = min_by_months(out, ["07", "08"], "NDVI")
    sep_oct_ndvi = mean_by_months(out, ["09", "10"], "NDVI")
    sep_oct_gndvi = mean_by_months(out, ["09", "10"], "GNDVI")
    sep_oct_ndre = mean_by_months(out, ["09", "10"], "NDRE")
    jul_aug_gndvi_min = min_by_months(out, ["07", "08"], "GNDVI")
    jul_aug_ndre_min = min_by_months(out, ["07", "08"], "NDRE")
    jul_aug_ndvi = mean_by_months(out, ["07", "08"], "NDVI")
    jul_aug_gndvi = mean_by_months(out, ["07", "08"], "GNDVI")
    jul_aug_psri = mean_by_months(out, ["07", "08"], "PSRI")
    sep_oct_psri = mean_by_months(out, ["09", "10"], "PSRI")

    if may_jun_ndvi is not None and jul_aug_ndvi_min is not None:
        out["NDVI_DROP_SUMMER"] = may_jun_ndvi - jul_aug_ndvi_min
    if sep_oct_ndvi is not None and jul_aug_ndvi_min is not None:
        out["NDVI_RECOVERY_AUTUMN"] = sep_oct_ndvi - jul_aug_ndvi_min
    if sep_oct_gndvi is not None and jul_aug_gndvi_min is not None:
        out["GNDVI_RECOVERY_AUTUMN"] = sep_oct_gndvi - jul_aug_gndvi_min
    if sep_oct_ndre is not None and jul_aug_ndre_min is not None:
        out["NDRE_RECOVERY_AUTUMN"] = sep_oct_ndre - jul_aug_ndre_min
    if sep_oct_ndvi is not None and jul_aug_ndvi is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_NDVI"] = sep_oct_ndvi - jul_aug_ndvi
    if sep_oct_gndvi is not None and jul_aug_gndvi is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_GNDVI"] = sep_oct_gndvi - jul_aug_gndvi
    if sep_oct_psri is not None and jul_aug_psri is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_PSRI"] = sep_oct_psri - jul_aug_psri

    ndvi_cols = month_columns(out, MODEL_MONTHS, "NDVI")
    gndvi_cols = month_columns(out, MODEL_MONTHS, "GNDVI")

    if ndvi_cols:
        out["ARGMIN_MONTH_NDVI"] = out[ndvi_cols].idxmin(axis=1).map(lambda x: int(x[4:6]) if pd.notna(x) else np.nan)
    if gndvi_cols:
        out["ARGMAX_MONTH_GNDVI"] = out[gndvi_cols].idxmax(axis=1).map(lambda x: int(x[4:6]) if pd.notna(x) else np.nan)

    return out


def keep_final_columns(df):
    """Csak a végleges modellhez használt oszlopokat tartja meg."""
    meta_cols = [c for c in META_COLS if c in df.columns]

    time_cols = [
        c for c in df.columns
        if len(c) >= 8 and c[:8].isdigit() and c[4:6] in MODEL_MONTHS and any(c.endswith(f"_{idx}") for idx in INDICES)
    ]
    stat_cols = [c for c in df.columns if c.startswith("STAT_")]
    diff_cols = [c for c in DIFF_FEATURES if c in df.columns]
    pheno_cols = [c for c in PHENO_FEATURES if c in df.columns]

    final_cols = meta_cols + sorted(time_cols) + sorted(stat_cols) + diff_cols + pheno_cols
    return df[final_cols].copy()


# 5. Futtatás

OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

print("Sentinel-2 kompozitok keresése...")
rasters, all_periods = find_composite_rasters()
model_periods = [p for p in all_periods if p[4:6] in MODEL_MONTHS]
ref_raster = get_reference_raster(rasters, all_periods)

print("Pixelértékek kinyerése...")
raw_pixels = extract_raw_pixels(rasters, all_periods, ref_raster)

print("Érvényes pixelek szűrése...")
raw_pixels = raw_pixels.loc[valid_rows(raw_pixels, model_periods)].reset_index(drop=True)

print("Sávértékek interpolációja...")
raw_pixels = interpolate_bands(raw_pixels, model_periods)

print("Vegetációs indexek és feature-k számítása...")
meta = raw_pixels[META_COLS].copy()
index_features = calculate_indices(raw_pixels, model_periods)
features = add_phenology(index_features)
features = add_index_statistics(features)
features = add_differences(features)
features = features.replace([np.inf, -np.inf], np.nan)
features = pd.concat([meta, features], axis=1)
features = keep_final_columns(features)

print("Parquet mentése...")
features.to_parquet(OUT_PARQUET, index=False)

print(f"KÉSZ: {OUT_PARQUET}")
print(f"Feature-mátrix mérete: {features.shape[0]} sor x {features.shape[1]} oszlop")


7. Random Forest modell tanítása

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit


# 1. Beállítások


MODEL_VERSION = "v1_2025_parlagfu"
EXPERIMENT_NAME = "vegleges_optimalizalt"

BASE_DIR = Path("/Users/petrascsaba/Library/CloudStorage/OneDrive-EotvosLorandTudomanyegyetemInformatikaiKar/Pál Márton fájljai - petras_csaba/03_output_data/01_RF_model")

FEATURE_PARQUET = BASE_DIR / "00_train/v1_2025_parlagfu.parquet"
LOG_DIR = BASE_DIR / "01_log"
MODEL_DIR = BASE_DIR / "02_model"
FI_DIR = BASE_DIR / "03_feature_importance"

ID_COL = "id"
TARGET_COL = "target_binary"
META_COLS = ["id", "pixel_id", "target", "target_binary"]

RANDOM_STATE = 42
TEST_SIZE = 0.20
PROBABILITY_THRESHOLD = 0.7

USE_SMOTE = False

USE_TOP_FI_SELECTION = True
TOP_N_FEATURES = 28
TOP_FI_CSV = FI_DIR / "FI_v1_2025_parlagfu_vegleges_top50.csv"

RF_N_ESTIMATORS = 300
RF_MAX_DEPTH = 20


WINDOW_CONFIG = {
    "months": ["05", "06", "07", "08", "09", "10"],
    "indices": ["NDVI", "GNDVI", "NDRE", "PSRI", "NDWI"],
    "stats": ["min", "mean", "max", "std", "amplitude"],
    "diffs": [
        "DIFF_NDVI_JUL_JUN", "DIFF_GNDVI_JUL_JUN", "DIFF_NDRE_JUL_JUN",
        "DIFF_NDVI_AUG_JUL", "DIFF_NDRE_AUG_JUL",
        "DIFF_NDVI_SEP_AUG", "DIFF_NDRE_SEP_AUG", "DIFF_PSRI_SEP_AUG",
        "RATIO_G_N_AUG", "RATIO_NDRE_NDVI_AUG",
    ],
    "phenology_features": [
        "NDVI_DROP_SUMMER", "NDVI_RECOVERY_AUTUMN", "GNDVI_RECOVERY_AUTUMN",
        "NDRE_RECOVERY_AUTUMN", "SUMMER_TO_AUTUMN_SLOPE_NDVI",
        "SUMMER_TO_AUTUMN_SLOPE_GNDVI", "SUMMER_TO_AUTUMN_SLOPE_PSRI",
        "ARGMIN_MONTH_NDVI", "ARGMAX_MONTH_GNDVI",
    ],
    "probability_threshold": PROBABILITY_THRESHOLD,
}


# 2. Segédfüggvények


def make_model_matrix(df):
    """Metaoszlopok eltávolítása, csak numerikus feature-k megtartása."""
    drop_cols = [c for c in META_COLS if c in df.columns]
    X = df.drop(columns=drop_cols)
    X = X.select_dtypes(include=[np.number])
    return X


def load_top_features(fi_csv, top_n, available_columns):
    """A top FI CSV első top_n sorából kiválasztja a modellben tényleg meglévő változókat."""
    if not fi_csv.exists():
        raise FileNotFoundError(f"Nem található feature importance CSV: {fi_csv}")

    fi = pd.read_csv(fi_csv, index_col=0)
    selected = []

    for feature_name in fi.index.tolist():
        if feature_name in available_columns:
            selected.append(feature_name)

        if len(selected) == top_n:
            break

    if len(selected) == 0:
        raise RuntimeError("A top FI CSV alapján egyetlen feature sem található a parquetben.")

    return selected


def predict_with_threshold(model, X, threshold):
    """0,7-es valószínűségi küszöb alkalmazása a pozitív osztályra."""
    prob = model.predict_proba(X)[:, 1]
    return (prob >= threshold).astype(int)


def calculate_metrics(y_true, y_pred):
    """Confusion matrix és klasszifikációs metrikák számítása."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        "precision_0": report["0"]["precision"],
        "recall_0": report["0"]["recall"],
        "f1_0": report["0"]["f1-score"],
        "precision_1": report["1"]["precision"],
        "recall_1": report["1"]["recall"],
        "f1_1": report["1"]["f1-score"],
        "accuracy": report["accuracy"],
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "FNR": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
    }

    return cm, report, metrics


def save_feature_importance(model, feature_names):
    """Feature importance mentése CSV-be és PNG-be."""
    FI_DIR.mkdir(parents=True, exist_ok=True)

    fi = pd.Series(
        model.feature_importances_,
        index=feature_names
    ).sort_values(ascending=False)

    fi_csv = FI_DIR / f"FI_{MODEL_VERSION}_{EXPERIMENT_NAME}_top50.csv"
    fi.to_frame("importance").head(50).to_csv(fi_csv)

    plt.rcParams["font.family"] = "Times New Roman"
    plt.figure(figsize=(10, 8))
    fi.head(28).sort_values().plot(kind="barh", color="#52BE80")
    plt.xlabel("Fontosság", fontsize=12, fontweight="bold")
    plt.ylabel("Jellemzők", fontsize=12, fontweight="bold")
    plt.grid(axis="x", linestyle="--", alpha=0.3)
    plt.tight_layout()

    fi_png = FI_DIR / f"FI_{MODEL_VERSION}_{EXPERIMENT_NAME}.png"
    plt.savefig(fi_png, dpi=300)
    plt.close()

    return fi_csv, fi_png, fi


def save_log(metrics, feature_names, model):
    """Egy soros CSV log mentése feature importance értékekkel együtt."""
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    fi = pd.Series(
        model.feature_importances_,
        index=feature_names
    ).sort_values(ascending=False)

    features_with_importance = ";".join([
        f"{name}={value:.4f}" for name, value in fi.items()
    ])

    row = {
        "version": MODEL_VERSION,
        "experiment_name": EXPERIMENT_NAME,
        "params": {"n_estimators": RF_N_ESTIMATORS, "max_depth": RF_MAX_DEPTH},
        "features_count": len(feature_names),
        "features_list": ";".join(feature_names),
        "features_importance": features_with_importance,
        "comment": "SUMMER_ONLY, threshold 0.7, SMOTE off",
    }

    row.update(metrics)

    log_path = LOG_DIR / f"{MODEL_VERSION}_{EXPERIMENT_NAME}_log.csv"
    pd.DataFrame([row]).to_csv(log_path, index=False)

    return log_path


def save_model(model, feature_names):
    """A modell és a szükséges metaadatok mentése joblib fájlba."""
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    bundle = {
        "model": model,
        "experiment_name": EXPERIMENT_NAME,
        "version": MODEL_VERSION,
        "window_config": WINDOW_CONFIG,
        "feature_names": feature_names,
        "probability_threshold": PROBABILITY_THRESHOLD,
        "rf_params": {
            "n_estimators": RF_N_ESTIMATORS,
            "max_depth": RF_MAX_DEPTH,
        },
    }

    model_path = MODEL_DIR / f"rf_{MODEL_VERSION}_{EXPERIMENT_NAME}.joblib"
    joblib.dump(bundle, model_path)

    return model_path



# 3. Futtatás


LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FI_DIR.mkdir(parents=True, exist_ok=True)

print("Feature parquet beolvasása...")
df = pd.read_parquet(FEATURE_PARQUET)
df = df.loc[df[TARGET_COL].notna()].reset_index(drop=True)

print("Tanító mátrix összeállítása...")
y = df[TARGET_COL].astype(int)
groups = df[ID_COL]
X = make_model_matrix(df)

if USE_TOP_FI_SELECTION:
    selected_features = load_top_features(
        TOP_FI_CSV,
        TOP_N_FEATURES,
        list(X.columns)
    )
    X = X[selected_features].copy()
    print(f"Top FI változók használata: {len(selected_features)} db")
else:
    selected_features = list(X.columns)
    print(f"Összes változó használata: {len(selected_features)} db")

# Hiányzó vagy végtelen értéket tartalmazó sorok eldobása
valid = np.isfinite(X).all(axis=1)
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)
groups = groups.loc[valid].reset_index(drop=True)

print(f"Tanításra használt pixelek száma: {len(X)}")
print(f"Felhasznált változók száma: {X.shape[1]}")

print("Train-test bontás GroupShuffleSplit segítségével...")
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Random Forest modell tanítása...")
model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("Teszt adathalmaz kiértékelése...")
y_pred = predict_with_threshold(
    model,
    X_test,
    PROBABILITY_THRESHOLD
)

cm, report, metrics = calculate_metrics(y_test, y_pred)

print("\nTévesztési mátrix:")
print(cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

fi_csv, fi_png, fi = save_feature_importance(model, selected_features)

print("\nFeature importance értékek:")
for name, value in fi.items():
    print(f"{name}: {value:.4f}")

log_path = save_log(metrics, selected_features, model)
model_path = save_model(model, selected_features)

print("\nKÉSZ")
print(f"Modell: {model_path}")
print(f"Log: {log_path}")
print(f"Feature importance CSV: {fi_csv}")
print(f"Feature importance PNG: {fi_png}")

8. Képosztályozás

In [ ]:
import glob
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.windows import Window


# 1. Beállítások

MODEL_VERSION = "v1_2025_parlagfu"
EXPERIMENT_NAME = "vegleges_optimalizalt"

BASE_DIR = Path("/Users/petrascsaba/Library/CloudStorage/OneDrive-EotvosLorandTudomanyegyetemInformatikaiKar/Pál Márton fájljai - petras_csaba/03_output_data/01_RF_model")

MODEL_PATH = BASE_DIR / f"02_model/rf_{MODEL_VERSION}_{EXPERIMENT_NAME}.joblib"
OUT_DIR = BASE_DIR / "04_classified"

COMPOSITE_DIR = Path("/Volumes/T7 Touch/06_szakdoga_sentinel/2025_merged_bb_10m_composites")
CLCPLUS_PATH = Path("/Volumes/T7 Touch/06_szakdoga_sentinel/egyéb/CLCplus/CLMS_CLCPLUS_RAS_S2023_R10m_V01_R00_merge_clip_align.tif")

BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
MODEL_MONTHS = ["05", "06", "07", "08", "09", "10"]
INDICES = ["NDVI", "GNDVI", "NDRE", "PSRI", "NDWI"]

DIFF_FEATURES = [
    "DIFF_NDVI_JUL_JUN", "DIFF_GNDVI_JUL_JUN", "DIFF_NDRE_JUL_JUN",
    "DIFF_NDVI_AUG_JUL", "DIFF_NDRE_AUG_JUL",
    "DIFF_NDVI_SEP_AUG", "DIFF_NDRE_SEP_AUG", "DIFF_PSRI_SEP_AUG",
    "RATIO_G_N_AUG", "RATIO_NDRE_NDVI_AUG",
]

PHENO_FEATURES = [
    "NDVI_DROP_SUMMER", "NDVI_RECOVERY_AUTUMN", "GNDVI_RECOVERY_AUTUMN",
    "NDRE_RECOVERY_AUTUMN", "SUMMER_TO_AUTUMN_SLOPE_NDVI",
    "SUMMER_TO_AUTUMN_SLOPE_GNDVI", "SUMMER_TO_AUTUMN_SLOPE_PSRI",
    "ARGMIN_MONTH_NDVI", "ARGMAX_MONTH_GNDVI",
]

VALID_CLC_CODES = [6, 7]

BLOCK_SIZE = 512
OUTPUT_NODATA = 255
OUTPUT_COMPRESS = "lzw"

MAX_BLOCKS = None #None-teljes térkép


# 2. Raszterek keresése és maszk

def find_composite_rasters():
    """A kompozit mappából összegyűjti a dátum-sáv párokat."""
    rasters = {}
    files = glob.glob(str(COMPOSITE_DIR / "*_composite.tif"))

    for file in files:
        stem = Path(file).stem
        parts = stem.split("_")
        period = parts[0]
        band = None

        for p in parts:
            if p in BANDS:
                band = p
                break

        if band is None or len(period) != 8 or not period.isdigit():
            continue

        month = period[4:6]
        if month in MODEL_MONTHS:
            if period not in rasters:
                rasters[period] = {}
            rasters[period][band] = file

    periods = sorted(rasters.keys())

    if not periods:
        raise RuntimeError("Nem találtam használható Sentinel-2 kompozitokat.")

    return rasters, periods


def get_reference_raster(rasters, periods):
    """Referencia raszter: lehetőleg B04, ha nincs, akkor az első elérhető sáv."""
    first_period = periods[0]

    if "B04" in rasters[first_period]:
        return rasters[first_period]["B04"]

    return list(rasters[first_period].values())[0]


def make_clc_mask(ref_path):
    """CLCplus 6 és 7 osztályok illesztése a Sentinel-2 rácsra."""
    with rasterio.open(ref_path) as ref:
        dst_shape = ref.shape
        dst_transform = ref.transform
        dst_crs = ref.crs

    with rasterio.open(CLCPLUS_PATH) as src:
        aligned = np.zeros(dst_shape, dtype=np.int16)

        reproject(
            source=rasterio.band(src, 1),
            destination=aligned,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest,
        )

    return np.isin(aligned, VALID_CLC_CODES)


def read_block_as_dataframe(rasters, periods, window, pixel_index):
    """Egy raszterblokk prediktálandó pixeleinek sávértékeit DataFrame-be olvassa."""
    raw = pd.DataFrame(index=np.arange(len(pixel_index)))

    for period in periods:
        for band in BANDS:
            col = f"{period}_{band}"
            path = rasters.get(period, {}).get(band)

            if path is None:
                raw[col] = np.nan
                continue

            with rasterio.open(path) as src:
                arr = src.read(1, window=window).reshape(-1)[pixel_index].astype("float32")

                if src.nodata is not None:
                    arr[arr == src.nodata] = np.nan

                raw[col] = arr / 10000.0

    return raw



# 3. Feature-előállítás
#    A tanító parquet készítésével egyező logika.


def interpolate_bands(raw, periods):
    """Hiányzó sávértékek lineáris interpolációja időben, sávonként."""
    out = raw.copy()

    for band in BANDS:
        cols = [f"{p}_{band}" for p in periods if f"{p}_{band}" in out.columns]

        if cols:
            out[cols] = out[cols].interpolate(
                axis=1,
                method="linear",
                limit_direction="both"
            )

    return out


def calculate_indices(raw, periods):
    """NDVI, GNDVI, NDRE, PSRI és NDWI számítása."""
    feat = pd.DataFrame(index=raw.index)

    with np.errstate(divide="ignore", invalid="ignore"):
        for p in periods:
            b2 = raw[f"{p}_B02"]
            b3 = raw[f"{p}_B03"]
            b4 = raw[f"{p}_B04"]
            b5 = raw[f"{p}_B05"]
            b6 = raw[f"{p}_B06"]
            b8 = raw[f"{p}_B08"]
            b11 = raw[f"{p}_B11"]

            feat[f"{p}_NDVI"] = (b8 - b4) / (b8 + b4)
            feat[f"{p}_GNDVI"] = (b8 - b3) / (b8 + b3)
            feat[f"{p}_NDRE"] = (b8 - b5) / (b8 + b5)
            feat[f"{p}_PSRI"] = (b4 - b2) / b6
            feat[f"{p}_NDWI"] = (b8 - b11) / (b8 + b11)

    return feat.replace([np.inf, -np.inf], np.nan)


def add_index_statistics(feat):
    """Indexenként min, max, átlag, szórás, amplitúdó."""
    out = feat.copy()

    for index_name in INDICES:
        cols = [c for c in out.columns if c.endswith(f"_{index_name}")]

        if not cols:
            continue

        values = out[cols]

        out[f"STAT_{index_name}_min"] = values.min(axis=1)
        out[f"STAT_{index_name}_mean"] = values.mean(axis=1)
        out[f"STAT_{index_name}_max"] = values.max(axis=1)
        out[f"STAT_{index_name}_std"] = values.std(axis=1)
        out[f"STAT_{index_name}_amplitude"] = values.max(axis=1) - values.min(axis=1)

    return out


def first_month_column(df, month, index_name):
    """Egy hónap első elérhető kompozitját adja vissza adott indexből."""
    cols = [
        c for c in df.columns
        if len(c) >= 8
        and c[:8].isdigit()
        and c[4:6] == month
        and c.endswith(f"_{index_name}")
    ]

    if not cols:
        return None

    return df[sorted(cols)[0]]


def add_differences(feat):
    """Fenológiai különbségek és arányok előállítása."""
    out = feat.copy()

    pairs = {
        "DIFF_NDVI_JUL_JUN": ("07", "06", "NDVI"),
        "DIFF_GNDVI_JUL_JUN": ("07", "06", "GNDVI"),
        "DIFF_NDRE_JUL_JUN": ("07", "06", "NDRE"),
        "DIFF_NDVI_AUG_JUL": ("08", "07", "NDVI"),
        "DIFF_NDRE_AUG_JUL": ("08", "07", "NDRE"),
        "DIFF_NDVI_SEP_AUG": ("09", "08", "NDVI"),
        "DIFF_NDRE_SEP_AUG": ("09", "08", "NDRE"),
        "DIFF_PSRI_SEP_AUG": ("09", "08", "PSRI"),
    }

    for name, (late_m, early_m, index_name) in pairs.items():
        late = first_month_column(out, late_m, index_name)
        early = first_month_column(out, early_m, index_name)

        if late is not None and early is not None:
            out[name] = late - early

    gndvi_aug = first_month_column(out, "08", "GNDVI")
    ndvi_aug = first_month_column(out, "08", "NDVI")
    ndre_aug = first_month_column(out, "08", "NDRE")

    if gndvi_aug is not None and ndvi_aug is not None:
        out["RATIO_G_N_AUG"] = gndvi_aug / (ndvi_aug + 1e-6)

    if ndre_aug is not None and ndvi_aug is not None:
        out["RATIO_NDRE_NDVI_AUG"] = ndre_aug / (ndvi_aug + 1e-6)

    return out


def month_columns(df, months, index_name):
    return [
        c for c in df.columns
        if len(c) >= 8
        and c[:8].isdigit()
        and c[4:6] in months
        and c.endswith(f"_{index_name}")
    ]


def mean_by_months(df, months, index_name):
    cols = month_columns(df, months, index_name)

    if not cols:
        return None

    return df[cols].mean(axis=1)


def min_by_months(df, months, index_name):
    cols = month_columns(df, months, index_name)

    if not cols:
        return None

    return df[cols].min(axis=1)


def add_phenology(feat):
    """Fenológiát leíró változók előállítása."""
    out = feat.copy()

    may_jun_ndvi = mean_by_months(out, ["05", "06"], "NDVI")
    jul_aug_ndvi_min = min_by_months(out, ["07", "08"], "NDVI")
    sep_oct_ndvi = mean_by_months(out, ["09", "10"], "NDVI")

    sep_oct_gndvi = mean_by_months(out, ["09", "10"], "GNDVI")
    jul_aug_gndvi_min = min_by_months(out, ["07", "08"], "GNDVI")

    sep_oct_ndre = mean_by_months(out, ["09", "10"], "NDRE")
    jul_aug_ndre_min = min_by_months(out, ["07", "08"], "NDRE")

    jul_aug_ndvi = mean_by_months(out, ["07", "08"], "NDVI")
    jul_aug_gndvi = mean_by_months(out, ["07", "08"], "GNDVI")
    jul_aug_psri = mean_by_months(out, ["07", "08"], "PSRI")
    sep_oct_psri = mean_by_months(out, ["09", "10"], "PSRI")

    if may_jun_ndvi is not None and jul_aug_ndvi_min is not None:
        out["NDVI_DROP_SUMMER"] = may_jun_ndvi - jul_aug_ndvi_min

    if sep_oct_ndvi is not None and jul_aug_ndvi_min is not None:
        out["NDVI_RECOVERY_AUTUMN"] = sep_oct_ndvi - jul_aug_ndvi_min

    if sep_oct_gndvi is not None and jul_aug_gndvi_min is not None:
        out["GNDVI_RECOVERY_AUTUMN"] = sep_oct_gndvi - jul_aug_gndvi_min

    if sep_oct_ndre is not None and jul_aug_ndre_min is not None:
        out["NDRE_RECOVERY_AUTUMN"] = sep_oct_ndre - jul_aug_ndre_min

    if sep_oct_ndvi is not None and jul_aug_ndvi is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_NDVI"] = sep_oct_ndvi - jul_aug_ndvi

    if sep_oct_gndvi is not None and jul_aug_gndvi is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_GNDVI"] = sep_oct_gndvi - jul_aug_gndvi

    if sep_oct_psri is not None and jul_aug_psri is not None:
        out["SUMMER_TO_AUTUMN_SLOPE_PSRI"] = sep_oct_psri - jul_aug_psri

    ndvi_cols = month_columns(out, MODEL_MONTHS, "NDVI")
    gndvi_cols = month_columns(out, MODEL_MONTHS, "GNDVI")

    if ndvi_cols:
        out["ARGMIN_MONTH_NDVI"] = out[ndvi_cols].idxmin(axis=1).map(
            lambda x: int(x[4:6]) if pd.notna(x) else np.nan
        )

    if gndvi_cols:
        out["ARGMAX_MONTH_GNDVI"] = out[gndvi_cols].idxmax(axis=1).map(
            lambda x: int(x[4:6]) if pd.notna(x) else np.nan
        )

    return out


def build_features(raw, periods):
    """A térképi predikcióhoz előállítja a modell bemeneti jellemzőit."""
    raw = interpolate_bands(raw, periods)

    feat = calculate_indices(raw, periods)
    feat = add_phenology(feat)
    feat = add_index_statistics(feat)
    feat = add_differences(feat)
    feat = feat.replace([np.inf, -np.inf], np.nan)

    return feat



# 4. Klasszifikáció


OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modell betöltése...")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Nem található modell: {MODEL_PATH}")

bundle = joblib.load(MODEL_PATH)

model = bundle["model"]
expected_features = bundle["feature_names"]
threshold = bundle.get("probability_threshold", 0.7)

print(f"Elvárt feature-ök száma: {len(expected_features)}")
print(f"Döntési küszöb: {threshold}")

print("Sentinel-2 kompozitok keresése...")
rasters, periods = find_composite_rasters()

print(f"Felhasznált időablakok száma: {len(periods)}")
print(f"Első időablakok: {periods[:5]}")
print(f"Utolsó időablakok: {periods[-5:]}")

ref_path = get_reference_raster(rasters, periods)

with rasterio.open(ref_path) as ref:
    rows, cols = ref.shape
    ref_nodata = ref.nodata

    meta = {
        "driver": "GTiff",
        "height": ref.height,
        "width": ref.width,
        "count": 1,
        "dtype": "uint8",
        "crs": ref.crs,
        "transform": ref.transform,
        "nodata": OUTPUT_NODATA,
        "compress": OUTPUT_COMPRESS,
        "BIGTIFF": "IF_SAFER",
    }

print("CLCplus maszk illesztése...")
clc_mask = make_clc_mask(ref_path)

mask_pixels = int(clc_mask.sum())
mask_area_km2 = mask_pixels * 100 / 1_000_000

print(f"Maszkon belüli pixelek száma: {mask_pixels}")
print(f"Maszkon belüli terület: {mask_area_km2:.2f} km²")

out_path = OUT_DIR / f"parlagfu_{MODEL_VERSION}_{EXPERIMENT_NAME}_SUMMER_ONLY.tif"

print("Térképi klasszifikáció indítása...")

total_mask_pixels = 0
total_valid_rows = 0
total_invalid_rows = 0
total_positive = 0
processed_blocks = 0
debug_printed = False

with rasterio.open(ref_path) as ref_src, rasterio.open(out_path, "w", **meta) as dst:
    for row in range(0, rows, BLOCK_SIZE):
        for col in range(0, cols, BLOCK_SIZE):
            window = Window(
                col,
                row,
                min(BLOCK_SIZE, cols - col),
                min(BLOCK_SIZE, rows - row)
            )

            r0 = int(window.row_off)
            r1 = int(window.row_off + window.height)
            c0 = int(window.col_off)
            c1 = int(window.col_off + window.width)

            clc_block = clc_mask[r0:r1, c0:c1]

            ref_block = ref_src.read(1, window=window)

            ref_valid = np.ones(ref_block.shape, dtype=bool)

            if ref_nodata is not None:
                ref_valid = ref_valid & (ref_block != ref_nodata)

            if np.issubdtype(ref_block.dtype, np.floating):
                ref_valid = ref_valid & np.isfinite(ref_block)

            predict_mask = clc_block & ref_valid

            out_block = np.full(ref_block.shape, OUTPUT_NODATA, dtype="uint8")

            if not predict_mask.any():
                dst.write(out_block, 1, window=window)
                continue

            pixel_index = np.flatnonzero(predict_mask.reshape(-1))

            raw = read_block_as_dataframe(
                rasters=rasters,
                periods=periods,
                window=window,
                pixel_index=pixel_index
            )

            features = build_features(raw, periods)

            missing = [c for c in expected_features if c not in features.columns]
            if missing:
                raise RuntimeError(
                    "Hiányzó feature oszlopok: "
                    + ", ".join(missing[:30])
                )

            X = features[expected_features].copy()
            valid_rows = np.isfinite(X).all(axis=1).to_numpy()

            pred = np.full(len(X), OUTPUT_NODATA, dtype="uint8")

            total_mask_pixels += len(X)
            total_valid_rows += int(valid_rows.sum())
            total_invalid_rows += int((~valid_rows).sum())

            if valid_rows.any():
                prob = model.predict_proba(X.loc[valid_rows])[:, 1]
                pred[valid_rows] = (prob >= threshold).astype("uint8")
                total_positive += int((pred == 1).sum())

            if not debug_printed:
                print("\nDEBUG első prediktált blokk:")
                print(f"Feature mátrix mérete: {features.shape}")
                print(f"X mérete: {X.shape}")
                print(f"NaN arány az X-ben: {X.isna().mean().mean():.4f}")
                print(f"Érvényes sorok: {int(valid_rows.sum())}/{len(valid_rows)}")
                print(f"Pozitív pixelek ebben a blokkban: {int((pred == 1).sum())}")
                debug_printed = True

            out_flat = out_block.reshape(-1)
            out_flat[pixel_index] = pred
            out_block = out_flat.reshape(ref_block.shape)

            dst.write(out_block, 1, window=window)

            processed_blocks += 1

            if MAX_BLOCKS is not None and processed_blocks >= MAX_BLOCKS:
                print("Gyors teszt vége: elértem a MAX_BLOCKS értéket.")
                break

        print(f"Sor kész: {row}/{rows}")

        if MAX_BLOCKS is not None and processed_blocks >= MAX_BLOCKS:
            break


print("\nKimeneti raszter ellenőrzése...")
with rasterio.open(out_path) as src:
    arr = src.read(1)
    vals, counts = np.unique(arr, return_counts=True)
    unique_dict = dict(zip(vals.tolist(), counts.tolist()))

    print("Pixelértékek gyakorisága:")
    print(unique_dict)
    print("Kimeneti profil:")
    print(src.profile)


print("\nÖSSZESÍTÉS")
print(f"Predikcióra kijelölt pixelek: {total_mask_pixels}")
print(f"Érvényes feature sorok: {total_valid_rows}")
print(f"Érvénytelen feature sorok: {total_invalid_rows}")
print(f"Pozitív pixelek: {total_positive}")

if total_valid_rows > 0:
    print(f"Pozitív arány az érvényes predikciókon belül: {total_positive / total_valid_rows * 100:.2f}%")

positive_area_km2 = total_positive * 100 / 1_000_000
print(f"Becsült pozitív terület: {positive_area_km2:.2f} km²")

print(f"\nKÉSZ: {out_path}")